#### Base Agents: Sentiment + Technical with PPO and SAC

- Sentiment PPO
- Sentiment SAC
- Technical PPO
- Technical SAC


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import SAC, PPO
from sentiment_enviroment import SentimentEnv
from technical_enviroment import TechnicalEnv
from agent_wrapper import *
import warnings
from custom_function import add_regime_indicator
warnings.filterwarnings("ignore")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Configuration

All parameters in one place for easy experimentation


In [3]:

# Sentiment Agent Environment
sentiment_env_config = {
    'vol_window': 6,           # Rolling window for Sharpe-like reward (6 months)
    'transaction_cost': 0.002, # INCREASED from 0.001 (0.2% per trade)
    'verbose': 0,              # Silent during training
}

# Technical Agent Environment
technical_env_config = {
    'vol_window': 6,           # Rolling window for Sharpe-like reward (6 months)
    'transaction_cost': 0.002, # INCREASED from 0.001 (0.2% per trade)
    'verbose': 0,              # Silent during training
}

ppo_config = {
    # Learning rate - REDUCED for stability
    'learning_rate': 0.0005,   # REDUCED from 0.001 (slower learning)
    
    # Data collection
    'n_steps': 2048,           # Steps per rollout (keep as is)
    'batch_size': 256,          # REDUCED from 128 (more noise = less overfit)
    'n_epochs': 3,             # REDUCED from 5 (less data reuse)
    
    # Exploration - INCREASED to prevent premature convergence
    'ent_coef': 0.1,           # INCREASED from 0.05 (more exploration)
    
    # Discount and advantage estimation
    'gamma': 0.95,             # Discount factor (focus on near-term rewards)
    'gae_lambda': 0.9,         # GAE parameter (bias-variance tradeoff)
    
    # Policy update constraints
    'clip_range': 0.2,         # PPO clipping range (standard)
    'clip_range_vf': None,     # Value function clipping (None = no clip)
    
    # Loss function weights
    'vf_coef': 0.5,            # Value function coefficient
    
    # Gradient stability
    'max_grad_norm': 0.5,      # REDUCED gradient clipping (more stable)
    
    # Normalization
    'normalize_advantage': True,  # Normalize advantages (reduces variance)
    
    # Other
    'use_sde': False,          # State-dependent exploration (optional)
    'sde_sample_freq': -1,     # SDE sampling frequency
    'target_kl': None,         # Early stopping based on KL divergence (optional)
    
    # Verbosity
    'verbose': 1,
}

sac_config = {
    # Learning rate - REDUCED for stability
    'learning_rate': 0.0003,   # REDUCED from 0.001
    
    # Replay buffer - INCREASED for diversity
    'buffer_size': 50000,     # INCREASED from 50000 (more diverse experiences)
    'learning_starts': 700,   # INCREASED from 500 (more random exploration first)
    
    # Training
    'batch_size': 256,         # REDUCED from 256 (less overfitting)
    'train_freq': 1,           # Update every step
    'gradient_steps': 1,       # One gradient step per env step
    
    # Exploration - FIXED at higher value
    'ent_coef': 0.2,           # INCREASED from 'auto' (force more exploration)
    
    # Soft update
    'tau': 0.005,              # Soft update coefficient (slow target updates)
    
    # Discount factor
    'gamma': 0.95,             # Focus on near-term rewards
    
    # Network update
    'target_update_interval': 1,  # Update target network every step
    'target_entropy': 'auto',  # Automatic entropy target
    
    # Other
    'use_sde': False,          # State-dependent exploration
    'sde_sample_freq': -1,
    'use_sde_at_warmup': False,
    
    # Verbosity
    'verbose': 1,
}
training_config = {
    # Data split
    'train_ratio': 0.60,       # 60% for training
    'val_ratio': 0.20,         # 20% for validation
    
    # Training budget - REDUCED to prevent overfitting
    'timesteps': 100000,        # REDUCED from 100000 (stop before overfitting)
    
    # Algorithm selection
    'algorithm': 'both',       # 'ppo', 'sac', or 'both' - Train both algorithms to compare
    
    # Model saving
    'save_path': './models/',  # Directory for models and plots
}

early_stopping_config = {
    'eval_freq': 1500,         # Evaluate every 2000 steps (15 evals for 30K steps)
    'patience': 5,             # Stop if no improvement for 5 evaluations (10K steps)
    'min_delta': 0.01,          # Any improvement counts (let model learn)
}

# Super Agent Reward Function
super_agent_config = {
    'alpha_returns': 3.0,      # REDUCED from 5.0 (less reward hacking)
    'alpha_mdd': 1.0,          # INCREASED from 0.7 (penalize drawdowns more)
    'alpha_vol': 0.5,          # INCREASED from 0.2 (penalize volatility)
    'exploration_bias': 0.02,  # INCREASED from 0.01 (more exploration)
}

# Meta Agent Reward Function
meta_agent_config = {
    'alpha_returns': 3.0,      # REDUCED from 5.0
    'alpha_mdd': 1.0,          # INCREASED from 0.5
    'alpha_vol': 0.8,          # INCREASED from 0.5 (more volatility penalty)
    'exploration_bias': 0.02,  # INCREASED from 0.01
}

# Regime Detection
regime_config = {
    'enabled': True,           # Enable regime indicators
    'window': 6,               # SMA window for regime (6 months)
}

def print_config_summary():
    """Print configuration summary"""

    print("ANTI-OVERFITTING CONFIGURATION SUMMARY")

    print("\n Base Agents:")
    print(f"  Transaction Cost: {sentiment_env_config['transaction_cost']:.3f} (0.2%)")
    print(f"  Volatility Window: {sentiment_env_config['vol_window']} months")
    
    print("\n PPO Settings:")
    print(f"  Learning Rate: {ppo_config['learning_rate']}")
    print(f"  Batch Size: {ppo_config['batch_size']}")
    print(f"  Epochs: {ppo_config['n_epochs']}")
    print(f"  Entropy Coef: {ppo_config['ent_coef']} (HIGH exploration)")
    print(f"  Max Grad Norm: {ppo_config['max_grad_norm']}")
    
    print("\n SAC Settings:")
    print(f"  Learning Rate: {sac_config['learning_rate']}")
    print(f"  Buffer Size: {sac_config['buffer_size']:,}")
    print(f"  Batch Size: {sac_config['batch_size']}")
    print(f"  Entropy Coef: {sac_config['ent_coef']} (FIXED high)")
    print(f"  Learning Starts: {sac_config['learning_starts']}")
    
    print("\n Training:")
    print(f"  Max Timesteps: {training_config['timesteps']:,}")
    print(f"  Algorithm: {training_config['algorithm'].upper()}")
    print(f"  Early Stopping: Every {early_stopping_config['eval_freq']} steps")
    print(f"  Patience: {early_stopping_config['patience']} evaluations")



print_config_summary()


ANTI-OVERFITTING CONFIGURATION SUMMARY

 Base Agents:
  Transaction Cost: 0.002 (0.2%)
  Volatility Window: 6 months

 PPO Settings:
  Learning Rate: 0.0005
  Batch Size: 256
  Epochs: 3
  Entropy Coef: 0.1 (HIGH exploration)
  Max Grad Norm: 0.5

 SAC Settings:
  Learning Rate: 0.0003
  Buffer Size: 50,000
  Batch Size: 256
  Entropy Coef: 0.2 (FIXED high)
  Learning Starts: 700

 Training:
  Max Timesteps: 100,000
  Algorithm: BOTH
  Early Stopping: Every 1500 steps
  Patience: 5 evaluations


In [6]:
price_data = pd.read_csv("input_data/monthly_prices.csv", index_col=0, parse_dates=True)
technical_features = pd.read_csv("input_data/technical_indicators.csv", index_col=0, parse_dates=True)
sentiment_features = pd.read_csv("input_data/nlp_features.csv", index_col=0, parse_dates=True)
common_dates = technical_features.index.intersection(sentiment_features.index)
price_data = price_data.loc[common_dates]
technical_features = technical_features.loc[common_dates]
sentiment_features = sentiment_features.loc[common_dates]

price_data.to_csv("output_data/price_data.csv")
technical_features.to_csv("output_data/technical_features.csv")
sentiment_features.to_csv("output_data/sentiment_features.csv")
# Check data quality
print(f"Total months: {len(price_data)}")
print(f"NaN in prices: {price_data.isnull().sum().sum()}")
print(f"Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}")

Total months: 129
NaN in prices: 0
Date range: 2015-02-28 to 2025-10-31


## Add Market Regime Indicators

These indicators help super and meta agents adapt to bull/bear market conditions.

NOTE: Regime indicators are calculated on the full dataset before splitting. This is acceptable since:
- The SMA uses only past prices (no look-ahead bias)
- The indicator at time t only uses prices up to time t
- However, for maximum rigor, you could calculate regimes separately for train/val/test after splitting


In [8]:

# Add regime indicators for super and meta agents
# 1=bull market (price > SMA), 0=bear market (price < SMA)
if regime_config['enabled']:
    regime_indicators = add_regime_indicator(price_data, window=regime_config['window'])
    
    print(f"Regime indicators shape: {regime_indicators.shape}")
    #print(f"Regime indicators columns: {regime_indicators.columns.tolist()}")
    print(f"\nRecent market regimes:")
    #print(regime_indicators.tail(3))
    
    # Summary statistics
    bull_pct = (regime_indicators == 1).sum().sum() / regime_indicators.size * 100
    print(f"\nBull market periods: {bull_pct:.1f}%")
    print(f"Bear market periods: {100-bull_pct:.1f}%")
else:
    regime_indicators = None
    print("Regime indicators disabled")

regime_indicators.to_csv("output_data/regime_indicators.csv")

Regime indicators shape: (129, 10)

Recent market regimes:

Bull market periods: 60.7%
Bear market periods: 39.3%


In [7]:
# Train with the anti-overfitting configuration
results, summary_df, training_histories = train_and_evaluate_with_split(
    price_data=price_data,
    technical_features=technical_features,
    sentiment_features=sentiment_features,
    
    # Data split
    train_ratio=training_config['train_ratio'],
    val_ratio=training_config['val_ratio'],
    
    # Training parameters
    timesteps=training_config['timesteps'],
    algorithm=training_config['algorithm'],
    
    # Environment parameters
    sentiment_env_params=sentiment_env_config,
    technical_env_params=technical_env_config,
    
    # RL algorithm parameters
    ppo_params=ppo_config,
    sac_params=sac_config,
    
    # Early stopping with improved config
    early_stopping_params=early_stopping_config,
    save_path=training_config['save_path'],
)




DATA SPLITTING

Total data: 129 months
Date range: 2015-02-28 00:00:00 to 2025-10-31 00:00:00

DATA SPLIT SUMMARY:
Train:  77 months (60%) | 2015-02 to 2021-06
Val:    25 months (20%) | 2021-07 to 2023-07
Test:   27 months (20%) | 2023-08 to 2025-10

No data leakage - splits are properly separated

TRAINING SENTIMENT AGENTS

[Sentiment PPO]

Training Sentiment with PPO...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

Validation (Callback) Results:
  Sharpe Ratio:    0.101
  Total Return:   -3.26%
  Max Drawdown:   29.80%
  Win Rate:       50.00%
Step 1500: Val Sharpe = 0.101
  New best! Saved model (Sharpe: 0.101)
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 76       |
|    ep_rew_mean     | 30.4     |
| time/              |          |
|    fps             | 4123     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
----------------------

In [6]:
# ========================================================================
# STEP 4: Analyze Results
# ========================================================================

print("\n" + "="*70)
print("STEP 4: Results Analysis")
print("="*70)

# Display summary DataFrame
print("\n📊 Performance Summary:")
print(summary_df.to_string(index=False))

# Check which models generalize best
print("\n🏆 Best Generalizing Agent:")
best_agent = None
best_test_sharpe = -float('inf')

for name, res in results.items():
    test_sharpe = res['test_metrics']['sharpe_ratio']
    if test_sharpe > best_test_sharpe:
        best_test_sharpe = test_sharpe
        best_agent = name

print(f"  {best_agent.replace('_', ' ').title()}")
print(f"  Test Sharpe Ratio: {best_test_sharpe:.3f}")


STEP 4: Results Analysis

📊 Performance Summary:
        Agent  Train Sharpe  Val Sharpe  Test Sharpe  Train Return  Test Return
Sentiment Ppo      1.902096    0.718883     1.863048     13.484903     1.704953
Sentiment Sac      1.759351    0.176912     1.389680      9.456042     1.142117
Technical Ppo      2.150337    0.630232     1.929630     25.158725     1.675406
Technical Sac      1.931118    0.129822     1.358881     11.574541     1.167121

🏆 Best Generalizing Agent:
  Technical Ppo
  Test Sharpe Ratio: 1.930


In [ ]:

print("\n" + "="*70)
print("STEP 5: Generating Visualizations")
print("="*70)

# Plot comprehensive comparison
fig = plot_train_val_test_comparison(results)
comparison_path = f"{training_config['save_path']}/comparison_plot.png"
fig.savefig(comparison_path, dpi=150, bbox_inches='tight')
print(f"✓ Comparison plot saved: {comparison_path}")
plt.close()

# Plot validation curves for each agent
print("\n📈 Validation Curves (already saved during training):")
for name, callback in training_histories.items():
    if hasattr(callback, 'best_sharpe'):
        agent_name = name.replace('_', ' ').title()
        print(f"  {agent_name}:")
        print(f"    Best Val Sharpe: {callback.best_sharpe:.3f}")
        print(f"    Stopped at step: {callback.stopped_step or training_config['timesteps']}")


STEP 5: Generating Visualizations
✓ Comparison plot saved: ./models//comparison_plot.png

📈 Validation Curves (already saved during training):
  Sentiment Ppo:
    Best Val Sharpe: 0.719
    Stopped at step: 15000
  Sentiment Sac:
    Best Val Sharpe: 0.177
    Stopped at step: 9000
  Technical Ppo:
    Best Val Sharpe: 0.630
    Stopped at step: 24000
  Technical Sac:
    Best Val Sharpe: 0.130
    Stopped at step: 10500


In [ ]:
print("\n" + "="*70)
print("STEP 6: Models Saved")
print("="*70)

print("\n✓ Best models (selected by validation performance) saved in:")
for name in results.keys():
    model_path = f"{training_config['save_path']}/{name}/best_model.zip"
    print(f"  {model_path}")


STEP 6: Models Saved

✓ Best models (selected by validation performance) saved in:
  ./models//sentiment_ppo/best_model.zip
  ./models//sentiment_sac/best_model.zip
  ./models//technical_ppo/best_model.zip
  ./models//technical_sac/best_model.zip


In [9]:

print("\n" + "="*70)
print("STEP 7: Overfitting Diagnosis")
print("="*70)

def diagnose_overfitting(results):
    """Provide detailed overfitting diagnosis"""
    
    for name, res in results.items():
        train_sharpe = res['train_metrics']['sharpe_ratio']
        val_sharpe = res['val_metrics']['sharpe_ratio']
        test_sharpe = res['test_metrics']['sharpe_ratio']
        
        train_return = res['train_metrics']['total_return']
        test_return = res['test_metrics']['total_return']
        
        agent_name = name.replace('_', ' ').title()
        
        print(f"\n{agent_name}:")
        print(f"  Train Sharpe: {train_sharpe:>7.3f}")
        print(f"  Val Sharpe:   {val_sharpe:>7.3f}")
        print(f"  Test Sharpe:  {test_sharpe:>7.3f}")
        
        # Calculate various degradation metrics
        train_test_deg = (train_sharpe - test_sharpe) / abs(train_sharpe) * 100 if train_sharpe != 0 else 0
        val_test_deg = (val_sharpe - test_sharpe) / abs(val_sharpe) * 100 if val_sharpe != 0 else 0
        
        print(f"\n  Degradation Analysis:")
        print(f"    Train → Test: {train_test_deg:>6.1f}%")
        print(f"    Val → Test:   {val_test_deg:>6.1f}%")
        
        # Diagnosis
        print(f"\n  Diagnosis:")
        if train_test_deg < 20:
            print("    ✓ EXCELLENT - Model generalizes well")
        elif train_test_deg < 40:
            print("    ⚠ ACCEPTABLE - Moderate overfitting")
            print("    Recommendation: Consider increasing entropy_coef or transaction_cost")
        else:
            print("    ✗ OVERFITTING - Significant degradation")
            print("    Recommendation: Reduce timesteps, increase regularization")
        
        # Check if validation worked well
        if abs(val_test_deg) < 10:
            print("    ✓ Validation is predictive of test performance")
        else:
            print("    ⚠ Gap between validation and test")

diagnose_overfitting(results)


STEP 7: Overfitting Diagnosis

Sentiment Ppo:
  Train Sharpe:   1.480
  Val Sharpe:     0.422
  Test Sharpe:    2.127

  Degradation Analysis:
    Train → Test:  -43.8%
    Val → Test:   -404.6%

  Diagnosis:
    ✓ EXCELLENT - Model generalizes well
    ⚠ Gap between validation and test

Sentiment Sac:
  Train Sharpe:   1.727
  Val Sharpe:     0.129
  Test Sharpe:    1.164

  Degradation Analysis:
    Train → Test:   32.6%
    Val → Test:   -800.9%

  Diagnosis:
    ⚠ ACCEPTABLE - Moderate overfitting
    Recommendation: Consider increasing entropy_coef or transaction_cost
    ⚠ Gap between validation and test

Technical Ppo:
  Train Sharpe:   1.963
  Val Sharpe:     0.699
  Test Sharpe:    1.464

  Degradation Analysis:
    Train → Test:   25.4%
    Val → Test:   -109.4%

  Diagnosis:
    ⚠ ACCEPTABLE - Moderate overfitting
    Recommendation: Consider increasing entropy_coef or transaction_cost
    ⚠ Gap between validation and test

Technical Sac:
  Train Sharpe:   1.798
  Val Sharp

In [10]:

print("\n" + "="*70)
print("STEP 8: Next Steps & Recommendations")
print("="*70)

# Calculate average degradation
all_degradations = []
for name, res in results.items():
    train_sharpe = res['train_metrics']['sharpe_ratio']
    test_sharpe = res['test_metrics']['sharpe_ratio']
    deg = (train_sharpe - test_sharpe) / abs(train_sharpe) * 100 if train_sharpe != 0 else 0
    all_degradations.append(deg)

avg_degradation = sum(all_degradations) / len(all_degradations)

print(f"\n📊 Average Overfitting: {avg_degradation:.1f}%")

if avg_degradation < 20:
    print("\n✓ EXCELLENT RESULTS!")
    print("  Your models generalize well. Consider:")
    print("  • Using these models in production")
    print("  • Testing on more recent/different data")
    print("  • Exploring ensemble methods")
    
elif avg_degradation < 40:
    print("\n⚠ ACCEPTABLE BUT CAN IMPROVE")
    print("  Further reduce overfitting by:")
    print("  • Increase entropy_coef to 0.15-0.2")
    print("  • Reduce timesteps to 20k-25k")
    print("  • Increase transaction_cost to 0.003")
    print("  • Try smaller networks (64x64)")
    
else:
    print("\n✗ OVERFITTING DETECTED")
    print("  Immediate actions needed:")
    print("  • Reduce timesteps to 15k-20k")
    print("  • Increase entropy_coef to 0.2")
    print("  • Increase transaction_cost to 0.003-0.005")
    print("  • Reduce batch_size to 32 (PPO)")
    print("  • Set n_epochs to 2 (PPO)")
    print("  • Consider collecting more training data")




STEP 8: Next Steps & Recommendations

📊 Average Overfitting: 9.4%

✓ EXCELLENT RESULTS!
  Your models generalize well. Consider:
  • Using these models in production
  • Testing on more recent/different data
  • Exploring ensemble methods


## Hierarchical Training: Super Agent

Super Agent learns to blend Sentiment and Technical agents' recommendations



The Super Agent learns to optimally combine recommendations from the Sentiment and Technical agents.

**Hierarchical Architecture:**
- Base agents (Sentiment & Technical) analyze their specialized features
- Super Agent observes both base agents' portfolio weights
- Super Agent makes final portfolio allocation decisions

**Key Features:**
- Uses best performing base agents (highest test Sharpe)
- Learns adaptive blending strategy
- Can incorporate market regime indicators
- Benefits from diverse information sources

**Goal:** Achieve better risk-adjusted returns than individual base agents through intelligent combination.



In [12]:
# Super Agent configuration
super_agent_config = {
    'alpha_returns': 5.0,      # Weight for portfolio returns
    'alpha_mdd': 0.5,          # Weight for maximum drawdown penalty
    'alpha_vol': 0.5,          # Weight for volatility penalty
    'exploration_bias': 0.01   # Small exploration bonus
}



    """
    Example workflow for training hierarchical RL agents.  Replace the
    placeholder file paths and dataset loading logic to adapt this
    function for your own project.  This main function demonstrates the
    typical flow:

        1. Load raw data (prices, technical indicators, sentiment features).
        2. Split into train/validation/test sets chronologically.
        3. Instantiate base environments for sentiment and technical data.
        4. Load pre‑trained base agents (sentiment and technical).
        5. Build SuperAgentEnv instances and train the super agent.
        6. Build MetaAgentEnv instances and train the meta agent.
        7. Evaluate the performance of each agent on the test set.

    Note: Without access to the actual data files and pre‑trained model
    checkpoints, this example cannot be executed as is.  Use it as a
    reference when integrating your own data and models.
    """